# Densité de Population GHSL 2020 - Portugal

In [9]:
import ee
import geemap
from IPython.display import Image, display

In [10]:
ee.Initialize(project='YOUR_GEE_PROJECT')

In [ ]:
# Charger l'image GHSL Population 2020
ghsl_pop = ee.Image('JRC/GHSL/P2023A/GHS_POP/2020')


portugal_bbox = ee.Geometry.Rectangle([-9.5, 36.8, -6.2, 42.2])


population = ghsl_pop.select('population_count').clip(portugal_bbox)


population = population.updateMask(population.gt(0))

print("Image chargée et découpée pour le Portugal.")

Image chargée et découpée pour le Portugal.


In [ ]:

stats = population.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=portugal_bbox,
    scale=100,
    maxPixels=1e13
)

stats_dict = stats.getInfo()
print("=== Statistiques GHSL Population 2020 - Portugal ===")
for k, v in stats_dict.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

=== Statistiques GHSL Population 2020 - Portugal ===
  population_count_max: 1551.9330
  population_count_mean: 6.9459
  population_count_min: 0.0000


In [ ]:

vis_params = {
    'min': 0,
    'max': 1000,
    'palette': ['000004', '320A5A', '781B6C', 'BB3654', 'EC6824', 'FBB41A', 'FCFFA4']
}

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.addLayer(population, vis_params, 'GHSL Population 2020 - 100m')
Map.centerObject(portugal_bbox, 7)
Map

Map(center=[39.47664252846841, -7.849999999999961], controls=(WidgetControl(options=['position', 'transparent_…

In [14]:
# Générer et afficher une miniature PNG
thumbnail_params = {
    'min': 0,
    'max': 1000,
    'palette': ['000004', '320A5A', '781B6C', 'BB3654', 'EC6824', 'FBB41A', 'FCFFA4'],
    'dimensions': 512,
    'region': portugal_bbox,
    'format': 'png'
}

thumbnail_url = population.getThumbURL(thumbnail_params)
print("URL miniature :", thumbnail_url)
display(Image(url=thumbnail_url))

URL miniature : https://earthengine.googleapis.com/v1/projects/YOUR_GEE_PROJECT/thumbnails/b0a29a2e3c0db1c80c2b11d47caf46d0-bacaac0c47952e1439bde1a109e2d769:getPixels


In [15]:
# Export vers Google Drive en GeoTIFF
task = ee.batch.Export.image.toDrive(
    image=population,
    description='GHSL_Population_2020_Portugal',
    folder='GEE_Exports',
    fileNamePrefix='ghsl_pop_portugal_2020_100m',
    region=portugal_bbox,
    scale=100,                          # Résolution native GHSL
    maxPixels=1e13,
    crs='EPSG:4326'
)

task.start()
print("Export lancé !")
print("Statut :", task.status())

Export lancé !
Statut : {'state': 'READY', 'description': 'GHSL_Population_2020_Portugal', 'priority': 100, 'creation_timestamp_ms': 1776425700207, 'update_timestamp_ms': 1776425700207, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'YK6IEL3F2V4AFSWBLDGO57PX', 'name': 'projects/YOUR_GEE_PROJECT/operations/YK6IEL3F2V4AFSWBLDGO57PX'}
